<a href="https://colab.research.google.com/github/wilmar-barragan/PROCESAMIENTO-DE-LENGUAJE-NATURAL-TRANSFORMA-TEXTOS-EN-EMBEDDINGS-/blob/main/TRANSFORMA_TEXTOS_EN_EMBEDDINGS_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import re
import unicodedata

def preprocess_text(text: str) -> str:
    """Pipeline de preprocesamiento para embeddings."""
    # Normalización Unicode (NFC)
    text = unicodedata.normalize('NFC', text)
    # Eliminación de espacios redundantes
    text = re.sub(r'\s+', ' ', text).strip()
    # Opcional: eliminación de caracteres especiales según el caso de uso
    # NOTA: Para embeddings modernos, se preserva la puntuación
    # ya que los tokenizadores la manejan internamente.
    return text

In [22]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
tokens = tokenizer.tokenize("Transformar texto en embeddings es fundamental en PLN.")
# ['Trans', '##formar', 'texto', 'en', 'em', '##bed', '##ding', '##s', ...]

In [23]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Carga del modelo (se descarga de HuggingFace Hub la primera vez)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Textos de ejemplo
texts = [
    "La inteligencia artificial transforma la industria.",
    "El machine learning revoluciona los negocios.",
    "Me gusta comer pizza los viernes."
]

# Generación de embeddings (batch processing con normalización)
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Normalización L2 para similitud coseno directa
)

print(f"Forma del array de embeddings: {embeddings.shape}")
# Salida: (3, 384) → 3 textos, 384 dimensiones

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Forma del array de embeddings: (3, 384)


README.md

Notebook 01: Introducción a Embeddings
Contenido:
Introducción teórica con referencias bibliográficas.
Visualización de one-hot vs. embedding denso.
Carga de embeddings pre-entrenados (GloVe 100d) para comparación.
Demostración de analogías: rey - hombre + mujer ≈ reina.
Visualización 2D de un subconjunto de palabras.

Notebook 02: Embeddings con Transformers

Contenido:
Instalación y carga de sentence-transformers.
Comparación de tres modelos:
all-MiniLM-L6-v2 (384 dim, rápido, multilingüe básico)
paraphrase-multilingual-MiniLM-L12-v2 (384 dim, multilingüe)
text-embedding-3-small vía API OpenAI (1536 dim)
Generación de embeddings para un corpus de 50 documentos.
Análisis de tiempos de inferencia y consumo de memoria.
Almacenamiento en formato .npy para reutilización.

Notebook 03: Similitud y Aplicaciones

Contenido:
Cálculo de matrices de similitud coseno.
Visualización con heatmaps y reducción t-SNE/UMAP.
Implementación de búsqueda semántica (retrieval).
Análisis de clusters temáticos.
Caso práctico: sistema de preguntas frecuentes con matching semántico.


In [24]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_one_hot_vs_embedding(word: str, vocab: list, dim: int = 50):
    """Compara visualmente one-hot con embedding denso."""
    idx = vocab.index(word)

    one_hot = np.zeros(len(vocab))
    one_hot[idx] = 1.0

    # Embedding simulado (en práctica se cargaría de GloVe/Word2Vec)
    embedding = np.random.randn(dim)

    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    axes[0].bar(range(min(50, len(vocab))), one_hot[:50], color='steelblue')
    axes[0].set_title(f'One-Hot: "{word}" (dim={len(vocab)})')
    axes[0].set_xlabel('Posición en vocabulario')

    axes[1].bar(range(dim), embedding, color='coral')
    axes[1].set_title(f'Embedding denso: "{word}" (dim={dim})')
    axes[1].set_xlabel('Dimensión')

    plt.tight_layout()
    plt.savefig('results/plots/one_hot_vs_embedding.png', dpi=150, bbox_inches='tight')
    plt.show()

In [27]:
import time
from sentence_transformers import SentenceTransformer

models_config = {
    'MiniLM-L6-v2': 'all-MiniLM-L6-v2',
    'Paraphrase-Multilingual': 'paraphrase-multilingual-MiniLM-L12-v2',
}

results = {}

# Define corpus_texts, for example, from an existing variable like 'texts'
corpus_texts = [
    "La inteligencia artificial transforma la industria.",
    "El machine learning revoluciona los negocios.",
    "Me gusta comer pizza los viernes."
]

for name, model_id in models_config.items():
    print(f"\n{'='*50}")
    print(f"Modelo: {name} ({model_id})")
    print(f"{'='*50}")

    # Carga del modelo
    start = time.time()
    model = SentenceTransformer(model_id)
    load_time = time.time() - start
    print(f"Tiempo de carga: {load_time:.2f}s")

    # Generación de embeddings
    start = time.time()
    embeddings = model.encode(
        corpus_texts,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True
    )
    inference_time = time.time() - start

    results[name] = {
        'embeddings': embeddings,
        'shape': embeddings.shape,
        'load_time': load_time,
        'inference_time': inference_time,
        'params_M': sum(p.numel() for p in model[0].auto_model.parameters()) / 1e6
    }

    print(f"Forma: {embeddings.shape}")
    print(f"Tiempo inferencia ({len(corpus_texts)} textos): {inference_time:.3f}s")
    print(f"Parámetros: {results[name]['params_M']:.1f}M")


Modelo: MiniLM-L6-v2 (all-MiniLM-L6-v2)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Tiempo de carga: 1.52s
Forma: (3, 384)
Tiempo inferencia (3 textos): 0.060s
Parámetros: 22.7M

Modelo: Paraphrase-Multilingual (paraphrase-multilingual-MiniLM-L12-v2)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Tiempo de carga: 5.04s
Forma: (3, 384)
Tiempo inferencia (3 textos): 0.075s
Parámetros: 117.7M


In [28]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query: str, corpus_embeddings: np.ndarray,
                    corpus_texts: list, model, top_k: int = 5) -> list:
    """
    Realiza búsqueda semántica sobre un corpus pre-indexado.

    Args:
        query: Texto de consulta.
        corpus_embeddings: Matriz (n_docs, d) de embeddings del corpus.
        corpus_texts: Lista de textos del corpus.
        model: Modelo SentenceTransformer para codificar la consulta.
        top_k: Número de resultados a retornar.

    Returns:
        Lista de tuplas (texto, score) ordenadas por relevancia.
    """
    # Codificar la consulta
    query_embedding = model.encode([query], normalize_embeddings=True)

    # Calcular similitud coseno con todo el corpus
    similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

    # Ordenar por similitud descendente
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = [(corpus_texts[i], float(similarities[i])) for i in top_indices]
    return results

# Ejemplo de uso
# Aseguramos que corpus_embeddings y model estén definidos para este contexto
# Usamos los resultados del último modelo procesado (Paraphrase-Multilingual) del cell YyitG1UGLF7Y
corpus_embeddings = results['Paraphrase-Multilingual']['embeddings']
# The 'model' variable from the loop in cell YyitG1UGLF7Y is still in scope

query = "¿Cómo funciona el aprendizaje automático?"
results = semantic_search(query, corpus_embeddings, corpus_texts, model, top_k=5)

print(f"\n🔍 Consulta: \"{query}\"\n")
print(f"{'Rank':<5}{'Score':<10}{'Documento'}")
print("-" * 70)
for rank, (text, score) in enumerate(results, 1):
    print(f"{rank:<5}{score:<10.4f}{text[:60]}...")


🔍 Consulta: "¿Cómo funciona el aprendizaje automático?"

Rank Score     Documento
----------------------------------------------------------------------
1    0.5632    El machine learning revoluciona los negocios....
2    0.4284    La inteligencia artificial transforma la industria....
3    -0.0062   Me gusta comer pizza los viernes....
